#####Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Which provides the richest feature set with field verification , discription and nested structure
**By default, Pydantic runs in Lax mode, performing automatic data parsing rather than just strict checking.If you pass a string "123" into an int field, Pydantic safely converts it to the integer 123

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')

model = init_chat_model('groq:qwen/qwen3-32b')
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001D1AC08C440>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001D1AC08CEC0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie") 
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")

In [3]:
model_with_structured=model.with_structured_output(Movie)
model_with_structured

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002236B9A1FD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002236B9A2A50>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movie rating out 

In [ ]:
model.invoke("Provide details about the movie titanic")# default response

AIMessage(content='<think>\nOkay, I need to provide details about the movie Titanic. Let me start by recalling what I know. It\'s a 1997 film directed by James Cameron, right? The main actors are Leonardo DiCaprio and Kate Winslet. They play Jack and Rose, two people from different social classes who fall in love on the ill-fated RMS Titanic. The movie is both a romance and a historical drama.\n\nFirst, the plot. The story is set in 1912. Rose is a wealthy woman engaged to a wealthy businessman, Cal Hockley, played by Billy Zane. She meets Jack, an artist with no money, and they form a bond. The ship hits an iceberg, leading to the sinking. Jack sacrifices himself so Rose can survive. I think the movie alternates between the romance and the disaster, showing the ship\'s luxurious aspects and the chaos of the sinking.\n\nThe film is known for its historical accuracy in depicting the Titanic, but it also has artistic liberties. For example, the heart-shaped necklace that Rose wears is fi

In [5]:
response = model_with_structured.invoke("provide details about the movie titanic")
response

Movie(title='Titanic', year=1997, director='James Cameron', rating=7.8)

Message output alongside parsed structure

In [7]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """A Movie with details."""
    title:str=Field(...,description="The title of the movie")
    year:int=Field(...,description="This year the movie was released")
    director:str=Field(...,description="The director of the movie")
    rating:float=Field(...,description="The movie rating out of 10")
    
model_with_strutured_row = model.with_structured_output(Movie,include_raw=True)
response = model_with_strutured_row.invoke("Provide me the details of the movie titanic")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the details of the movie Titanic. Let me check the functions available. There\'s a Movie function that requires title, year, director, and rating. I need to make sure I have all these details for Titanic. I know the title is "Titanic", the year it was released was 1997, the director is James Cameron, and the rating is 7.8. Let me confirm these facts to be accurate. Once I have all the required parameters, I can structure the JSON object with those details. I should make sure the types are correct: title and director as strings, year as an integer, and rating as a number. Alright, putting it all together in the tool_call format.\n', 'tool_calls': [{'id': 'h3m28w0hk', 'function': {'arguments': '{"director":"James Cameron","rating":7.8,"title":"Titanic","year":1997}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 199, 'prompt_tokens': 232, 

### Nested Structure

In [8]:
from pydantic import BaseModel,Field
class Actor(BaseModel):
    name:str
    role:str
class MovieDetails(BaseModel):
    title:str
    year: int
    cast:list[Actor]
    genere : list[str]
    budget : float|None = Field(description="budgets in millions usd")

model_with_nested_stucture = model.with_structured_output(MovieDetails)
respnse2=model_with_nested_stucture.invoke("provide me details about the movie titanic")
respnse2

MovieDetails(title='Titanic', year=1997, cast=[Actor(name='Leonardo DiCaprio', role='Jack Dawson'), Actor(name='Kate Winslet', role='Rose DeWitt Bukater'), Actor(name='Billy Zane', role='Cal Hockley'), Actor(name='Kathy Bates', role="Margaret 'Molly' Brown"), Actor(name='Johnny Depp', role="Caledon 'Cal' Hockley (as a ghost)")], genere=['Romance', 'Disaster'], budget=200.0)

### TypeDict
provides simpler alternative using python's built in, idial when you dont need runtime validation

In [9]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A Movie with details."""
    title:Annotated[str,...,"The title of the Movie"]
    year: Annotated[int,...,"Year of the movie released"]
    director: Annotated[str,...,"The director of the Movie"]
    ratings:Annotated[float,...,"The movie's rating out of 10"]

model_with_typedict = model.with_structured_output(MovieDict)
response3 = model_with_typedict.invoke("provide me the details of the movie avengers")
response3

{'director': 'Joss Whedon', 'ratings': 8, 'title': 'Avengers', 'year': 2012}

In [ ]:
model.profile # what are the thigs model has

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

#### Data Classes
Containing maily data, although there arent really any restrictions you create in a @dataclass decorator

In [2]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    '''Contact information for a person.'''
    name: str
    email: str
    phone: str

agent = create_agent(
    model = model,
    response_format=ContactInfo # auto select provided format
)
result = agent.invoke(
    {
        "messages":[{"role":"user","content":"Extract contact information from: Pavan,pavan@gmail.com,7733221166"}]
    }
)
result

{'messages': [HumanMessage(content='Extract contact information from: Pavan,pavan@gmail.com,7733221166', additional_kwargs={}, response_metadata={}, id='32dea10d-3acb-408a-b35e-0be8be01f169'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact information from the string "Pavan,pavan@gmail.com,7733221166". The tools provided include a ContactInfo function that requires name, email, and phone. \n\nFirst, I need to split the input into the three components. The input is comma-separated, so splitting by commas makes sense. The first part is the name, which is "Pavan". The second part is the email, "pavan@gmail.com", and the third is the phone number, "7733221166". \n\nI should check if all required fields are present. The required fields are name, email, and phone. All three are there. The email looks valid, and the phone number is a 10-digit number, which is typical for a US number. \n\nNow, I need to structure this int

In [3]:
result['structured_response']

ContactInfo(name='Pavan', email='pavan@gmail.com', phone='7733221166')